# InstantMesh on a **free** GPU (Google Colab / Kaggle)

This notebook runs the **official** `TencentARC/InstantMesh` pipeline (Zero123++ multi-view diffusion →
LRM sparse-view transformer → FlexiCubes mesh extraction) and writes a **real** GLB and OBJ.

It exists because the Arena sandbox used for the audit has **no NVIDIA GPU and no egress to
huggingface.co**, so real inference cannot happen there. This notebook is the cheapest verified way to
get a genuine image-to-3D asset for **$0**.

| | |
|---|---|
| Runtime | Colab free **T4 16 GB** (Kaggle **P100/T4 16 GB**, 30 GPU-h/week) |
| First run | ~15–25 min (torch + nvdiffrast compile + ~7.3 GB of weights) |
| Each image after warm-up | ~10–30 s on an A10G/A100; slower on T4 (≈1–3 min) |
| Output | `mesh.glb`, `mesh.obj`, multi-view grid, optional turntable MP4 |

**Do not use a CPU runtime** — `InstantMesh` hard-codes `torch.device('cuda')`.

## 1 · Check the GPU

In [ ]:
!nvidia-smi
import torch, sys
print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > Hardware accelerator > T4 GPU"
props = torch.cuda.get_device_properties(0)
print("device:", props.name, f"{props.total_memory/1024**3:.1f} GiB")

## 2 · Install the official dependencies

Order matters: install CUDA-enabled torch **before** `requirements.txt`, or pip will resolve a CPU-only build.
`nvdiffrast` compiles from source (5–10 min, this is normal).

If the torch pin fails on a newer Colab image, the matching pair from
<https://pytorch.org/get-started/locally/> is the safe substitute.

In [ ]:
%%bash
set -e
cd /content
[ -d InstantMesh ] || git clone https://github.com/TencentARC/InstantMesh.git
cd InstantMesh
python -m pip install -q --upgrade pip
python -m pip install -q torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121
python -m pip install -q xformers==0.0.22.post7
python -m pip install -q -r requirements.txt
echo "---- done ----"

## 3 · Fetch the weights (~7.3 GB total, auto-cached)

Sizes and SHA-256 from the Hub API (`TencentARC/InstantMesh`, verified 2026-11-01):

| file | bytes | GiB | sha256 |
|---|---|---|---|
| `instant_mesh_large.ckpt` | 1,514,818,077 | 1.41 | `329f7aae…03b91a1` |
| `instant_mesh_base.ckpt` | 1,253,574,354 | 1.17 | `22701cd2…6c44b5` |
| `diffusion_pytorch_model.bin` (white-bg UNet) | 1,732,056,502 | 1.61 | `9d78d784…fa3c4` |

Plus `sudo-ai/zero123plus-v1.2` (~5.4 GB incl. fp32 weights) and `facebook/dino-vitb16` (~0.33 GB).
`hf_hub_download` verifies the LFS hashes for you.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

ckpt = hf_hub_download("TencentARC/InstantMesh", "instant_mesh_large.ckpt", repo_type="model")
unet = hf_hub_download("TencentARC/InstantMesh", "diffusion_pytorch_model.bin", repo_type="model")
snapshot_download("sudo-ai/zero123plus-v1.2")
snapshot_download("facebook/dino-vitb16")
print(ckpt); print(unet)

## 4 · Run inference on your own image

Upload an image when Colab prompts, or use one of the repo's `examples/`.
Prefer `instant-mesh-base` if the T4 runs out of memory (see the VRAM note in
`SANDBOX_INSTANTMESH_AUDIT.md` — ~15 GiB peak is normal for `large`).

In [ ]:
import os, glob
CONFIG = "configs/instant-mesh-large.yaml"   # switch to instant-mesh-base.yaml if you hit CUDA OOM
os.chdir("/content/InstantMesh")

from google.colab import files           # Kaggle: comment out, use /kaggle/input instead
uploaded = files.upload()
name = next(iter(uploaded))
print("input:", name)

In [ ]:
%%bash
cd /content/InstantMesh
# --export_texmap adds UV unwrap (xatlas) and a texture map; slower, nicer for engines.
python run.py configs/instant-mesh-large.yaml examples/fox.jpg \
    --output_path outputs --seed 42 --diffusion_steps 75 --save_video

### Prefer your uploaded file
Set `INPUT=/content/InstantMesh/<your_file>` and re-run the cell above.
`run.py` writes OBJ only; the next cell converts to GLB (which is what the web app and most
engines want).

## 5 · Convert to GLB and download

In [ ]:
import glob, trimesh, os
os.chdir("/content/InstantMesh")
objs = sorted(glob.glob("outputs/*/meshes/*.obj"), key=os.path.getmtime)
print("OBJ files:", objs)
src = objs[-1]
mesh = trimesh.load(src, process=False)
print("faces:", len(mesh.faces), "vertices:", len(mesh.vertices), "visual:", type(mesh.visual).__name__)
glb = src.replace(".obj", ".glb")
mesh.export(glb)
print("wrote", glb, os.path.getsize(glb), "bytes")

from google.colab import files
files.download(glb)
files.download(src)
# multi-view grid (what the diffusion model actually produced):
for p in glob.glob("outputs/*/images/*.png"):
    files.download(p)

## 6 · Sanity checks — is this a *real* asset?

* `len(mesh.faces) > 0` and the GLB validates in <https://github.khronos.org/glTF-Validator/>.
* The multi-view PNG shows 6 consistent views of your object; if it looks broken, change `--seed`.
* Limitations to expect: thin/transparent parts (wires, glass), text on surfaces, and objects that are
  heavily occluded in the input. Back-view hallucination is normal for single-image methods.

In [ ]:
import trimesh
m = trimesh.load("/content/InstantMesh/outputs/instant-mesh-large/meshes/mesh.obj")
print("watertight:", m.is_watertight, "| bounds:", m.bounds.tolist())